# Build your own animal

This is a standalone first step for participants bringing their own keypoint data. The main teaching path starts with `01_intro_dmd.ipynb` and then moves to `02_bird_flight_dmd.ipynb`.

Use this notebook if you need to define a custom skeleton and check that your animation works before running DMD. Your keypoint data should be shaped `(n_frames, n_markers, 3)`.

We will define the marker order, the display polygons, the analysis markers, and a first `Animal3D` object that animates correctly.

### Setup

Run the next cell once in Colab. If you are running locally, install dependencies from the repo root with `uv sync` and skip notebook package installs.


In [ ]:
# Run this once at the start of Colab.
!pip install -q uv
!uv pip install --system -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

### Importing libraries

These help us build the skeleton and animate it. 

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import plotly.io as pio

try:
    from google.colab import files
except ImportError:
    files = None

from morphing_birds import Animal3D, SkeletonDefinition, animate_plotly

if "google.colab" in sys.modules:
    pio.renderers.default = "colab"

np.set_printoptions(precision=3, suppress=True)


## 1. Start with data shaped `(frames, markers, xyz)`

Here is a synthetic example. The first variable we need is a list of strings `marker_names = ["marker1","marker2",...]` which refer to the order in your dataset. 

Replace this synthetic example with your own loaded array. The important rule is that axis 1 must match `marker_names` exactly.


In [ ]:

# ---- FROM YOUR DATA ----

# marker_names = []

# ---- SYNTHETIC VERSION ----

marker_names = [
    "nose",
    "neck",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_front_foot",
    "right_front_foot",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_back_foot",
    "right_back_foot",
    "tail_base",
    "tail_tip",
    "tracking_tag",
]


For this example I am going to define my own custom animal and its motion data. You can load your data here instead.

In [ ]:

# --- FROM YOUR DATA ---

# motion = np.load(...)



# ---- SYNTHETIC VERSION ----
# A small top-down example: x is left/right, y is nose/tail, z is height.
mean_pose = np.array(
    [
        [0.00, 1.35, 0.00],   # nose
        [0.00, 0.95, 0.00],   # neck
        [-0.35, 0.70, 0.00],  # left_shoulder
        [0.35, 0.70, 0.00],   # right_shoulder
        [-0.60, 0.35, 0.00],  # left_elbow
        [0.60, 0.35, 0.00],   # right_elbow
        [-0.80, 0.05, 0.00],  # left_front_foot
        [0.80, 0.05, 0.00],   # right_front_foot
        [-0.30, -0.45, 0.00], # left_hip
        [0.30, -0.45, 0.00],  # right_hip
        [-0.52, -0.75, 0.00], # left_knee
        [0.52, -0.75, 0.00],  # right_knee
        [-0.70, -1.05, 0.00], # left_back_foot
        [0.70, -1.05, 0.00],  # right_back_foot
        [0.00, -0.70, 0.00],  # tail_base
        [0.00, -0.75, 0.00],  # tail_tip
        [0.00, 0.15, 0.25],   # tracking_tag
    ],
    dtype=float,
)

# I am going to invent some motion

n_frames = 60
phase = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
motion = np.repeat(mean_pose[None, :, :], n_frames, axis=0)

left_step = np.sin(phase)
right_step = np.sin(phase + np.pi)
motion[:, marker_names.index("left_front_foot"), 2] += 0.18 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_back_foot"), 2] += 0.14 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_front_foot"), 2] += 0.18 * np.maximum(right_step, 0)
motion[:, marker_names.index("left_back_foot"), 2] += 0.14 * np.maximum(right_step, 0)
motion[:, marker_names.index("tail_tip"), 0] += 0.08 * np.sin(phase)
motion[:, marker_names.index("neck"), 2] += 0.03 * np.sin(2 * phase)



Here is a checkpoint to see if your loading worked and my synthetic data is what we expect. 

In [ ]:

# ---- SANITY CHECK: MAKE SURE THIS RUNS! ----

assert motion.shape == (n_frames, len(marker_names), 3)
motion.shape


## 2. Define the skeleton

Now we have our markers defined and names, we need to build the polygons from the markers. For this week need to define body sections/parts and which markers define them. For this we need a dictionary `body_sections = {"head": ["marker1","marker2","marker3"]}` would for example make a triangle body part. 

If you want to include fixed keypoints we can do that here. These are points or parts of the body that don't move. in your data. For example, if you have wing keypoint data but you want to include a fixed body polygon that is in the animation/plots but not in the analysis we can declare some markers as `analysis_exclude`. 

Left/right pairs and centre markers make symmetry helpers explicit. So if there are markers with bilateral symmetry and others on the central line then you can differentiate them here.


In [ ]:
# --- FROM YOUR DATA ---

# body_sections = { "bodypart" : ["marker1", "marker2", "marker3"],
#                   "bodypart2" : ["marker1", "marker4", "marker5", "marker6"] }

# marker_pairs = [ ("left_marker1", "right_marker1"), 
#                  ("left_marker2", "right_marker2") ]

# centre_markers = ["centre_marker1"]



# ---- SYNTHETIC VERSION ----


body_sections = {
    "head": ["right_shoulder", "nose", "left_shoulder"],
    "torso": ["right_shoulder", "right_hip", "tail_base", "left_hip", "left_shoulder"],
    "tail": ["tail_base", "tail_tip"],
    "left_foreleg": ["left_shoulder", "left_elbow", "left_front_foot", "left_elbow", "left_shoulder"],
    "right_foreleg": ["right_shoulder", "right_elbow", "right_front_foot", "right_elbow", "right_shoulder"],
    "left_hindleg": ["left_hip", "left_knee", "left_back_foot", "left_knee", "left_hip"],
    "right_hindleg": ["right_hip", "right_knee", "right_back_foot", "right_knee", "right_hip"],
}

marker_pairs = [
    ("left_shoulder", "right_shoulder"),
    ("left_elbow", "right_elbow"),
    ("left_front_foot", "right_front_foot"),
    ("left_hip", "right_hip"),
    ("left_knee", "right_knee"),
    ("left_back_foot", "right_back_foot"),
]

centre_markers = ["nose", "neck", "tail_base", "tail_tip", "tracking_tag"]



## Now we can build the skeleton with the variables we have made. 

In [ ]:


skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"], # <-- the name of any fixed markers
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)

print("All markers:", skel.all_marker_names)
print("Analysis markers:", skel.analysis_markers)
print("Display-only markers:", skel.display_only_markers)


## 3. Build the animal and check the raw animation

With the skeleton built we can now plot and animate it. 

`mean_pose` is a static shape taken from the motion data and our first view of the animal. You can drag to rotate the plot and look at your animal. If you need to adjust the polygons you can go back up in the notebook to fix it. 

We can then include `motion` data with an animation. As you can see my custom animal is some kind of twitching flattened hedgehog. 

In [ ]:
mean_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=mean_pose)

fig = animate_plotly(animal, motion, axes_visible=True)
fig.show()


## 4. Handoff to the DMD notebook

The maths should use only the moving keypoints. Therefore we use the method `get_analysis_data` which will help us to automatically exclude markers we don't want.

For the DMD and other analyses like SVD/PCA you'll need a flat version of the motion data as well. 

In [ ]:
X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)

# -- DMD analysis happens here, see next notebook --


# -- You end up with a reconstruction. For now we will just copy the original. --

recon_flat = X_flat.copy()
recon = recon_flat.reshape(-1, X.shape[1], 3)


# -- Now we can animate it --
fig = animate_plotly(animal, recon, axes_visible=False)
fig.show()

print("DMD input:", X_flat.shape)
print("Animation reconstruction:", recon.shape)


## Save the skeleton for notebook 03

This saves only the skeleton definition, not your motion data. In Colab, the JSON file is also downloaded so you can upload it later if the runtime resets before you open `03_custom_dmd.ipynb`.

In [ ]:
skeleton_config = {
    "name": skel.name,
    "marker_names": list(marker_names),
    "body_sections": body_sections,
    "analysis_exclude": sorted(skel.analysis_exclude),
    "marker_pairs": [list(pair) for pair in marker_pairs],
    "centre_markers": list(centre_markers),
}

skeleton_path = Path("/content/custom_skeleton.json") if "google.colab" in sys.modules else Path("custom_skeleton.json")
skeleton_path.write_text(json.dumps(skeleton_config, indent=2))

print(f"Saved skeleton to {skeleton_path}")
if files is not None:
    files.download(str(skeleton_path))

## Reference: copyable skeleton setup

For the DMD notebook, the recommended handoff is the saved `custom_skeleton.json` file above. This full setup block is kept as a reference in case you want to copy the skeleton definition manually.


In [ ]:
marker_names = [
    "nose",
    "neck",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_front_foot",
    "right_front_foot",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_back_foot",
    "right_back_foot",
    "tail_base",
    "tail_tip",
    "tracking_tag",
]

# Replace this synthetic motion with your data loader.
mean_pose = np.array(
    [
        [0.00, 1.35, 0.00],
        [0.00, 0.95, 0.00],
        [-0.35, 0.70, 0.00],
        [0.35, 0.70, 0.00],
        [-0.60, 0.35, 0.00],
        [0.60, 0.35, 0.00],
        [-0.80, 0.05, 0.00],
        [0.80, 0.05, 0.00],
        [-0.30, -0.45, 0.00],
        [0.30, -0.45, 0.00],
        [-0.52, -0.75, 0.00],
        [0.52, -0.75, 0.00],
        [-0.70, -1.05, 0.00],
        [0.70, -1.05, 0.00],
        [0.00, -0.70, 0.00],
        [0.00, -1.35, 0.00],
        [0.00, 0.15, 0.25],
    ],
    dtype=float,
)
motion = np.repeat(mean_pose[None, :, :], 60, axis=0)
assert motion.ndim == 3 and motion.shape[1:] == (len(marker_names), 3)

body_sections = {
    "head": ["right_shoulder", "nose", "left_shoulder"],
    "torso": ["right_shoulder", "right_hip", "tail_base", "left_hip", "left_shoulder"],
    "tail": ["tail_base", "tail_tip"],
    "left_foreleg": ["left_shoulder", "left_elbow", "left_front_foot", "left_elbow", "left_shoulder"],
    "right_foreleg": ["right_shoulder", "right_elbow", "right_front_foot", "right_elbow", "right_shoulder"],
    "left_hindleg": ["left_hip", "left_knee", "left_back_foot", "left_knee", "left_hip"],
    "right_hindleg": ["right_hip", "right_knee", "right_back_foot", "right_knee", "right_hip"],
}
marker_pairs = [
    ("left_shoulder", "right_shoulder"),
    ("left_elbow", "right_elbow"),
    ("left_front_foot", "right_front_foot"),
    ("left_hip", "right_hip"),
    ("left_knee", "right_knee"),
    ("left_back_foot", "right_back_foot"),
]
centre_markers = ["nose", "neck", "tail_base", "tail_tip", "tracking_tag"]

skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"],
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)
mean_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=mean_pose)

X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)


## Troubleshooting

- Marker order mismatch: `marker_names[i]` must name `motion[:, i, :]`.
- Wrong units or axes: check whether your data is metres, centimetres, pixels, or whether vertical is `z`.
- Missing values: use `NaN` for unknown coordinates so `np.nanmean` does the right thing.
- Polygon marker typo: every name in `body_sections` must also appear in `marker_names`.
- Display-only markers: put fixed landmarks in `analysis_exclude`; moving keypoints usually belong in the analysis set.
